# 09 — Análise científica dos resultados

Análise local, somente CPU, dos arquivos já produzidos pelos notebooks 07 e 08. **Não carrega checkpoints, não treina nem executa inferência.** Lê CSVs/JSONs e confere métricas a partir das predições já salvas.

Execute as células em ordem no Jupyter local. Também é possível usar o Colab: a primeira célula monta o Drive, lê a pasta do experimento dos notebooks 07/08 e obtém o código atual em um clone temporário separado. Não depende de git pull ou de src/ atualizada no Drive. Instale o ambiente de análise descrito no README com `requirements-analysis.txt`. O ambiente não precisa de PyTorch, torchvision, CUDA ou acesso ao Google Drive. Não use o requirements completo de treinamento para esta tarefa.

As fontes permanecem intactas. Cada execução gera uma pasta nova em `reports/resultados/`, com tabelas CSV/LaTeX, figuras PDF/PNG, relatório Markdown e manifesto de hashes. Médias e desvios são entre as seeds 42/123/456, com **ddof=1**. A comparação é descritiva e não seleciona modelos ou hiperparâmetros pelo teste.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT_OVERRIDE = None  # raiz dos resultados: caminho local ou pasta do experimento no Drive

def find_project_root(candidates):
    unique = list(dict.fromkeys(Path(p).expanduser().resolve() for p in candidates))
    with_code = [p for p in unique if (p / 'src/analyze_results.py').is_file()]
    with_results = [p for p in with_code if (p / 'results/repetitions_comparison/validation_individual.csv').is_file()]
    if with_results:
        return with_results[0]
    if with_code:
        return with_code[0]  # a auditoria seguinte identificará os arquivos de resultados ausentes
    checked = '\n'.join(str(p / 'src/analyze_results.py') for p in unique)
    raise FileNotFoundError(
        'Não encontrei src/analyze_results.py nos caminhos abaixo:\n' + checked +
        '\nDefina PROJECT_ROOT_OVERRIDE para a raiz correta. '
        'Abrir o notebook pelo GitHub não sincroniza a pasta src/ no Drive. '
        'Se o projeto for um clone, atualize-o com git pull --ff-only nessa pasta; '
        'caso contrário, copie src/ atualizada para ela. Preserve results/. '
        'Depois reexecute esta célula. Não instale analyze_results com pip.'
    )

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount('/content/drive')

if IN_COLAB:
    import subprocess
    import tempfile
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE or '/content/drive/MyDrive/plant-disease-classification').expanduser().resolve()
    if not (PROJECT_ROOT / 'results').is_dir():
        raise FileNotFoundError(f'Pasta de resultados ausente: {PROJECT_ROOT / "results"}. Ajuste PROJECT_ROOT_OVERRIDE.')
    CODE_ROOT = Path(tempfile.mkdtemp(prefix='tcc-analise-')) / 'repo'
    cloned = subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/murilodc/plant-disease-classification.git', str(CODE_ROOT)],
        capture_output=True, text=True,
    )
    if cloned.returncode != 0:
        raise RuntimeError('Falha ao obter código em pasta temporária:\n' + cloned.stdout + cloned.stderr)
    if not (CODE_ROOT / 'src/analyze_results.py').is_file():
        raise FileNotFoundError(f'Script ausente no clone: {CODE_ROOT / "src/analyze_results.py"}')
else:
    if PROJECT_ROOT_OVERRIDE is not None:
        candidates = [PROJECT_ROOT_OVERRIDE]
    else:
        current = Path.cwd()
        candidates = [current, *current.parents, Path.home() / 'Documents/tcc-plant-disease-classification']
    PROJECT_ROOT = find_project_root(candidates)
    CODE_ROOT = PROJECT_ROOT
RESULTS_ROOT = PROJECT_ROOT / 'results'
TEST_DIR = None  # com várias avaliações completas, informe explicitamente a pasta do experimento
OUTPUT_DIR = None  # cria pasta nova em PROJECT_ROOT/reports/, fora de results/
sys.path.insert(0, str(CODE_ROOT / 'src'))
sys.modules.pop('analyze_results', None)  # evita reutilizar módulo antigo no mesmo kernel
print('Ambiente:', 'Colab (análise CPU)' if IN_COLAB else 'Jupyter local (análise CPU)')
print('Código:', CODE_ROOT)
print('Projeto com os resultados:', PROJECT_ROOT)
print('Resultados originais:', RESULTS_ROOT)


## Inventário e integridade

A auditoria exige 12 execuções por split, os mapas originais de classes e suportes consistentes. Confere os hashes dos artefatos do teste, matrizes contra predições salvas, métricas individuais e consolidadas, ddof=1, tempos e melhores épocas contra os históricos.

Os consolidados de `results/repetitions_comparison/` são a fonte de validação. Resultados individuais antigos de outra GPU não substituem os consolidados. A lista de originais históricos ausentes ou divergentes vai para o relatório. Diferenças de CRLF/LF são registradas sem modificar arquivos.

Se um arquivo necessário estiver ausente ou inconsistente, a execução para com o caminho ou a checagem específica. Se houver mais de uma avaliação final completa, selecione `TEST_DIR` pelo protocolo/proveniência; não escolha a avaliação pelo melhor resultado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
from IPython.display import display, Markdown, Image
from analyze_results import load_saved_results, scientific_tables, metric_display, markdown_table

print('Python:', sys.version.split()[0], '| pandas:', pd.__version__, '| NumPy:', np.__version__, '| matplotlib:', matplotlib.__version__)
data = load_saved_results(RESULTS_ROOT, TEST_DIR)
tables = scientific_tables(data)
print('Avaliação selecionada:', data.test_dir)
print('Artefatos de teste verificados:', len(data.manifest['artifacts_sha256']))
display(data.inventory)
display(data.historical_sources.groupby('status').size().rename('Arquivos').reset_index())
for note in data.notes:
    print(note)

## Validação: média ± desvio padrão amostral

Valores em porcentagem. O desvio usa a mesma escala percentual, em pontos percentuais. Cada linha reúne três execuções. As métricas macro atribuem peso igual a cada classe; as ponderadas usam o suporte de imagens.

In [ ]:
display(metric_display(tables['validation_summary']))
display(tables['validation_individual'][['model_name', 'seed', 'accuracy', 'f1_macro', 'f1_weighted']])

## Teste final: média ± desvio padrão amostral

8.125 imagens por execução. Os desvios quantificam a variação entre treinamentos, sem representar intervalos de confiança sobre novas imagens. Não há teste inferencial ou escolha de modelo nesta análise.

In [ ]:
display(metric_display(tables['test_summary']))
display(tables['test_individual'][['model_name', 'seed', 'accuracy', 'f1_macro', 'f1_weighted']])
display(tables['validation_test_differences'][['model_name', 'seed', 'accuracy_delta_pp', 'f1_macro_delta_pp']])

## Classes e confusões

Compare precisão, recall, F1 e suporte. As cinco classes com menores F1 são uma descrição posterior dos resultados, sem ajustes de treinamento. Contagens e taxas de confusão não são intercambiáveis: a taxa tem como denominador o suporte da classe real.

As matrizes normalizadas são promediadas entre sementes, sem tratar as passagens pelas mesmas imagens como amostras independentes. A validação consolidada não contém matrizes numéricas nem predições por imagem; não se extraem contagens de PNGs.

In [ ]:
display(tables['test_difficult_classes'][['model_name', 'class_index', 'class_name', 'support', 'precision_mean', 'recall_mean', 'f1_score_mean', 'f1_score_std']])
display(tables['test_confusions_top_count'])
display(tables['test_confusions_top_rate'])
display(tables['test_error_stability'])
display(tables['class_support'])

## Custo computacional

Tempos são os registrados nos experimentos originais na GPU, analisados no PC sem GPU. O total de treino inclui treino e validação por época e depende do early stopping. A latência por imagem é uma média de throughput end-to-end em batches, não latência de requisição isolada. Inferência de validação e teste permanece separada, inclusive por diferenças de aquecimento.

Não se deduzem FLOPs, memória ou consumo energético a partir da contagem de parâmetros.

In [ ]:
display(tables['cost_summary'])
display(tables['cost_individual'][['model_name', 'seed', 'gpu', 'pytorch_version', 'cuda_version', 'batch_size', 'num_workers', 'epochs_executed', 'training_time_minutes', 'average_time_per_epoch_seconds', 'parameters_millions', 'inference_milliseconds_per_image', 'test_latency_ms', 'warmup_batches']])

## Exportação para redação acadêmica

Gera tabelas em CSV e LaTeX com booktabs, oito figuras em PDF vetorial e PNG 300 dpi, relatório fundamentado nos valores lidos e manifesto de integridade. O diretório de destino deve ser novo e ficar fora de `results/`. Os hashes das fontes são conferidos novamente ao final.

In [ ]:
from analyze_results import run_analysis
OUTPUT_DIR = run_analysis(PROJECT_ROOT, test_dir=data.test_dir, output_dir=OUTPUT_DIR)
print('Saída:', OUTPUT_DIR)
print('Relatório:', OUTPUT_DIR / 'relatorio_resultados.md')
print('Tabelas:', OUTPUT_DIR / 'tables')
print('Figuras:', OUTPUT_DIR / 'figures')

In [ ]:
import os
relative = Path(os.path.relpath(OUTPUT_DIR, Path.cwd())).as_posix()
display(Markdown(f'[Abrir relatório Markdown]({relative}/relatorio_resultados.md)'))
for figure in ('01_metricas', '06_custo_computacional'):
    display(Image(filename=str(OUTPUT_DIR / 'figures' / (figure + '.png')), width=900))
report = (OUTPUT_DIR / 'relatorio_resultados.md').read_text(encoding='utf-8')
display(Markdown(report.split('## Integridade e arquivos ausentes')[0]))

## Leitura científica e limitações

- Maiores médias observadas não demonstram superioridade estatística. As barras são desvios entre três sementes, não intervalos de confiança.
- Uma divisão fixa e imagens do PlantVillage não estabelecem generalização para campo. Folhas distintas entre splits reduzem uma fonte de vazamento, sem eliminar todas as dependências possíveis.
- A baseline é treinada do zero e as redes de transferência partem de pré-treinamento. A comparação não isola o efeito causal da arquitetura.
- Suportes pequenos tornam métricas por classe sensíveis a poucos erros. Não se atribui a causa das confusões a sintomas ou qualidade visual sem inspeção própria.
- Consulte `historical_sources.csv` para saber quais originais faltam para auditoria completa. Os CSVs consolidados permitem esta análise sem carregar esses checkpoints.
- Qualquer hipótese sugerida pelo teste é uma observação posterior. Não altere retrospectivamente os modelos ou hiperparâmetros deste experimento.

Para o TCC, utilize os PDFs das figuras e as tabelas .tex compactas. O relatório inclui legendas, unidades, fontes locais, limites de interpretação e um exemplo de inclusão de figura no LaTeX.